# Substructure Importance

Notebook for running substructure importance experiments.

## Setup

In [ ]:
# imports
# standard libraries
import glob
import logging
import os
from pathlib import Path
import re
from string import ascii_lowercase
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml


from IPython.core.interactiveshell import InteractiveShell
from topological_pretraining.data import load_dataset
from topological_pretraining.data.mol import MorganGenerator, SortAndSlice
from topological_pretraining.featurization.pretrained import PreTrainedFeaturizer
from topological_pretraining.importance import gnn_importance
from topological_pretraining.models import LGBM
from matplotlib.collections import PolyCollection
from matplotlib.font_manager import FontProperties, findfont
from matplotlib.markers import MarkerStyle
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from matplotlib.collections import PolyCollection
from matplotlib.patches import FancyBboxPatch, Patch
from PIL import Image, ImageDraw, ImageFont
from rdkit import Chem
from scipy import stats
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    make_scorer,
    matthews_corrcoef,
)
from sklearn.metrics._scorer import (
    average_precision_scorer,
    neg_mean_absolute_error_scorer,
    r2_scorer,
    roc_auc_scorer,
)
from tqdm import tqdm

In [ ]:
# for custom paths, set environment variables in an .env file
# import dotenv
# dotenv.load_dotenv(override=False)

In [ ]:
# paths to directories

DATA_DIR = os.environ.get("DATA_DIR", False)
if DATA_DIR:
    DATA_DIR = Path(DATA_DIR)
    if not DATA_DIR.exists():
        raise ValueError(f"DATA_DIR {DATA_DIR} does not exist.")
else:
    DATA_DIR = Path("../data")
    if not DATA_DIR.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = os.environ.get("RESULTS_DIR", "../results")
RESULTS_DIR = Path(RESULTS_DIR)
if not RESULTS_DIR.exists():
    raise ValueError(f"RESULTS_DIR {RESULTS_DIR} does not exist.")

HYPERPARAMS_DIR = os.environ.get("HYPERPARAMS_DIR", "../data/hyperparameters")
HYPERPARAMS_DIR = Path(HYPERPARAMS_DIR)
if not HYPERPARAMS_DIR.exists():
    raise ValueError(f"HYPERPARAMS_DIR {HYPERPARAMS_DIR} does not exist.")

FIGURES_DIR = os.environ.get("FIGURES_DIR", "./figures")
FIGURES_DIR = Path(FIGURES_DIR)

(FIGURES_DIR / "biogen").mkdir(parents=True, exist_ok=True)
(FIGURES_DIR / "molnet").mkdir(parents=True, exist_ok=True)
(FIGURES_DIR / "chembl").mkdir(parents=True, exist_ok=True)
(FIGURES_DIR / "muv").mkdir(parents=True, exist_ok=True)
(FIGURES_DIR / "tox21").mkdir(parents=True, exist_ok=True)

TEMP_DIR = os.environ.get("TEMP_DIR", "../temp")
TEMP_DIR = Path(TEMP_DIR)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = os.environ.get("MODELS_DIR", "../pt_models")
MODEL_DIR = Path(MODEL_DIR)

LOGGING_LEVEL = int(os.environ.get("LOGGING_LEVEL", 20))
LOW_RES = bool(int(os.environ.get("LOW_RES", 1)))
IMG_SIZE_SCALE = float(os.environ.get("IMG_SIZE_SCALE", 1.0))
FILE_FORMAT = os.environ.get("FILE_FORMAT", "pdf")

In [ ]:
# path to save substructure importance results
# to do a fresh run, delete the existing directory and its contents
IMPORTANCE_RESULTS = RESULTS_DIR / "substructure_importance"
IMPORTANCE_RESULTS.mkdir(parents=True, exist_ok=True)

In [ ]:
def is_interactive():
    """
    True if running in an interactive Jupyter frontend (VS Code, JupyterLab, classic),
    False if running headless via nbconvert.
    """
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if not ip:
            return False

        if ip.__class__.__name__ != "ZMQInteractiveShell":
            return True

        # Try to locate the connection file
        connection_file = getattr(ip, 'connection_file', None)
        if not connection_file:
            # Sometimes passed as "-f /path/to/connection.json"
            for idx, arg in enumerate(sys.argv):
                if arg == "-f" and idx + 1 < len(sys.argv):
                    connection_file = sys.argv[idx + 1]
                    break
                if arg.endswith(".json") and os.path.isfile(arg):
                    connection_file = arg
                    break

        if not connection_file or not os.path.isfile(connection_file):
            return True

        # Check how recently the file was created/modified
        mtime = os.path.getmtime(connection_file)
        age_seconds = time.time() - mtime

        if age_seconds < 5:
            return False

        return True
    except Exception:
        return False

In [ ]:
if not is_interactive():
    (TEMP_DIR / "logs").mkdir(parents=True, exist_ok=True)
    all_logs = glob.glob(str(TEMP_DIR / "logs" / "*.log"))
    if len(all_logs) == 0:
        logger_path = str(TEMP_DIR / "logs" / "tmp0.log")
    else:
        numbers = [int(re.findall(r"\d+", l)[0]) for l in all_logs]
        last = max(numbers)
        logger_path = str(TEMP_DIR / "logs" / f"tmp{last + 1}.log")
    logging.basicConfig(
        level=LOGGING_LEVEL, 
        handlers=[
            logging.FileHandler(filename=logger_path, mode="w+"),
            logging.StreamHandler(sys.__stderr__),
            logging.StreamHandler(sys.__stdout__),
        ],
        format="[%(asctime)s] %(levelname)s: %(message)s", 
        datefmt="%Y-%m-%d %H:%M:%S"
    )
    logger = logging.getLogger(name="my_logger")
    logger.propagate = True
    logger.info("Logging started")
    logger.info(f"Logger path: {logger_path}")
else:
    logger_path = None
    logging.basicConfig(
        level=logging.INFO, 
        handlers=[
            logging.StreamHandler(sys.__stderr__),
            logging.StreamHandler(sys.__stdout__),
        ],
        format="[%(asctime)s] %(levelname)s: %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )
    logger = logging.getLogger(name="my_logger")
    logger.info("Logging started")

In [ ]:
logger.info(is_interactive())

In [ ]:
logger.info(f"Data path: {DATA_DIR}")
logger.info(f"Results path: {RESULTS_DIR}")
logger.info(f"Hyperparameters path: {HYPERPARAMS_DIR}")
logger.info(f"Figures path: {FIGURES_DIR}")
logger.info(f"Model path: {MODEL_DIR}")
logger.info(f"Low resolution: {LOW_RES}")

In [ ]:
# Exception Hook for Jupyter 
def notebook_exception_handler(shell, etype, evalue, tb, tb_offset=None):
    """
    This will run for uncaught exceptions in Jupyter cells.
    """
    logger.critical("HOOK TRIGGERED:", etype, evalue)
    logger.critical("Uncaught exception", exc_info=(etype, evalue, tb))

# Attach the handler to catch *all* exceptions
InteractiveShell.instance().set_custom_exc((Exception,), notebook_exception_handler)

In [ ]:
# set devices for models
logger.info("Setting devices for models")
torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lgbm_device = "gpu" if torch.cuda.is_available() else "cpu"

In [ ]:
# seaborn settings
logger.info("Setting seaborn settings")
sns.set(
    context="notebook",
    palette="deep",
)
sns.set_style("darkgrid", {"grid.color": ".6", "grid.linestyle": ":"})

In [ ]:
# list of datasets to run substructure importance experiment over
datasets_to_evaluate = [
    "FreeSolv", "Solu", "Human_CLint", 
    "SR_ARE", "MUV858"
]

In [ ]:
# dict of benchmark datasets and display names
benchmark_dataset_names = {
    # Biogen
    "Human_CLint": "Human CLint",
    "Solu": "Solubility",
    "FreeSolv":"FreeSolv",
    "MUV858":"MUV858",
    # Tox21
    "SR_ARE":"SR ARE",
}

In [ ]:
# load base standardizer settings
with open("../config/base/standardizer.yaml") as f:
    standardizer_kwargs = yaml.safe_load(f)["standardizer"]
    standardizer_kwargs["n_jobs"] = -1

In [ ]:
# loading datasets
logger.info("Loading datasets")
benchmark_datasets = {}
for i in benchmark_dataset_names:
    if i == "ESOL_restricted": continue
    logger.info(f"Loading dataset {i}")
    benchmark_datasets[i] = load_dataset(i, DATA_DIR, verbose=True, standardizer=standardizer_kwargs)


In [ ]:
runner = True # whether to run the importance calculations
save_figs = False # whether to save the figures generated by this notebook

## Load Results

Load results for selecting hyperparameters and models to use based on hp tuning results.

In [ ]:
# flatten and nest dictionary functions
def flatten_dict(d, sep=","):
    """
    Flatten a nested dictionary.
    """
    flat_dict = {}
    for key, value in d.items():
        if isinstance(value, dict):
            for sub_key, sub_value in flatten_dict(value, sep=sep).items():
                flat_dict[f"{key}{sep}{sub_key}"] = sub_value
        else:
            flat_dict[key] = value
    return flat_dict

def renest_dict(d, sep=","):
    """
    Renest a flattened dictionary.
    """
    nested_dict = {}
    for key, value in d.items():
        parts = key.split(sep)
        current_level = nested_dict
        for part in parts[:-1]:
            if part not in current_level:
                current_level[part] = {}
            current_level = current_level[part]
        current_level[parts[-1]] = value
    return nested_dict


In [ ]:
# load metrics results from file
logger.info("Load metrics for benchmark datasets")
results_path = RESULTS_DIR / "benchmark_results.npz"
if results_path.exists():
    results = np.load(results_path, allow_pickle=False,)
    results = renest_dict(results)
else:
    logger.warning(f"Results file {results_path} does not exist.")

## Functions for Sorting Methods

In [ ]:
logger.info("Setting up functions for sorting methods")

In [ ]:
# define splits for tuning
def get_tuning_splits():
    return np.arange(0, 5, dtype=int)

In [ ]:
# get array of metrics from results
def get_metrics(
    dataset, method, metric, splits = np.arange(0, 1000, dtype=int)
):
    try:
        if isinstance(splits, list):
            arr = np.array([results[dataset][method][metric][fold].mean() for fold in splits])
        else:
            arr = results[dataset][method][metric][splits]
    except Exception as e:
        print(f"Failed at:\n\tdataset = {dataset};\n\tmethod = {method};\n\tmetric = {metric}")
        raise ValueError(f"Error: {e}")
    if metric == "MAPE": arr *= 100
    return arr

In [ ]:
# get list of methods for particular dataset which match key
def find_methods(
    dataset, substring, 
    key = None,
):
    if key is None:
        key = lambda x, y: True if x.startswith(y) and ("filter" not in x) else False
    methods_list = list(results[dataset].keys())
    return [i for i in methods_list if key(i, substring)]

In [ ]:
# sorting ascending or descending
metric_sort_reverse = {
    "MAE": False,
    "AUCPR": True,
    "R2": True,
    "Pearson": True,
    "MAPE":False,
    "AUROC": True,
    "MCC": True,
    "EF10": True,
    "EF5": True,
    "EF1": True,
    "EF05": True,
}

In [ ]:
# get method to use for plotting
def get_method_to_plot(
    dataset,
    substring,
    metric = None,
    **kwargs 
):
    if metric is None:
        if benchmark_datasets[dataset].task == "regression":
            metric = "MAE"
        else:
            metric = "AUCPR"
    methods_list = find_methods(dataset, substring, **kwargs)
    tuning_means = {i: get_metrics(dataset, i, metric, get_tuning_splits()).mean() for i in methods_list}
    tuning_means = dict(sorted(tuning_means.items(), key=lambda i: i[1], reverse=metric_sort_reverse[metric]))
    best_method = next(iter(tuning_means.keys()))
    return best_method
    

## General Plotting Functions

In [ ]:
# renfer box around an axis
def render_box(
    fig, axes, 
    boxstyle="round,pad=0.01",
    edgecolor='grey',
    linestyle='dotted',          # dotted border
    linewidth=float(2*IMG_SIZE_SCALE),
    facecolor='none',            # transparent fill
    zorder=10,
    **kwargs
):
    positions = [ax.get_position() for ax in axes]
    renderer = fig.canvas.get_renderer()
    bboxes = [ax.get_tightbbox(renderer) for ax in axes]

    # Convert bboxes from display (pixel) to figure coordinates
    positions = [bbox.transformed(fig.transFigure.inverted()) for bbox in bboxes]
    x0 = min(pos.x0 for pos in positions)
    y0 = min(pos.y0 for pos in positions)
    x1 = max(pos.x1 for pos in positions)
    y1 = max(pos.y1 for pos in positions)
    # Create a FancyBboxPatch around the subplot
    box = FancyBboxPatch(
        (x0, y0), x1 - x0, y1 - y0,       # width and height
        boxstyle=boxstyle,   # rounded box
        edgecolor=edgecolor,
        linestyle=linestyle,          # dotted border
        linewidth=linewidth,
        facecolor=facecolor,            # transparent fill
        transform=fig.transFigure,   # use figure coords
        zorder=zorder,
        **kwargs,
    )
    # Add the patch to the figure
    fig.patches.append(box)

In [ ]:
if LOW_RES:
    dpi = 50
else:
    dpi = 300
logger.info(f"Setting seaborn savefig dpi to {dpi}")
sns.set_theme(
    rc={
        "savefig.dpi": dpi,
        "savefig.format": FILE_FORMAT,
    }
)

## Substructure Importance

### General Functions

In [ ]:
# display metrics for importance
metric_display_names = {
    "r2": "Importance (-$\\Delta\\mathrm{R^2})$",
    "aucpr": "Importance (-$\\Delta\\mathrm{AUCPR})$"
}

In [ ]:
# get molecules, y values, and splits for a dataset
def setup_dataset(dataset, split_idx=range(5,10)):
    _df = benchmark_datasets[dataset]
    molecules = _df.rdkit_mols
    y = _df.y.values
    splits = list(_df.splits)
    splits = [splits[i] for i in split_idx]
    return molecules, y, splits

In [ ]:
# get scorers for running feature importance
metric_scorers = {
    "classification": {
        "mcc": make_scorer(
                lambda y_true, y_pred: matthews_corrcoef(y_true, (y_pred > 0.5).astype(int)),
                response_method="predict_proba"
            ),
        "aucpr": average_precision_scorer,
        "auroc": roc_auc_scorer,   
    },
    "regression": {
        "neg_mae": neg_mean_absolute_error_scorer,
        "r2_score": r2_scorer,
        "pearsonr": make_scorer(lambda y_true, y_pred: stats.pearsonr(y_true, y_pred)[0],),
    }
}

In [ ]:
# get hyperparameters for a dataset and method
# method = "sns" for ECFP sort and slice
# method = "ecfp" for ECFP hashed
# method = "pt_gin" for pre-trained GIN
# method = "fcfp" for FCFP hashed
def get_hyperparameters(dataset, method):
    best_hp_method = get_method_to_plot(dataset, method) # get the best method in hp tuning
    hyperparameters_path = [i for i in HYPERPARAMS_DIR.rglob(f"*{best_hp_method}/{dataset}.yaml")]
    if len(hyperparameters_path) > 1: 
        raise ValueError(f"Too many paths for {best_hp_method} and {dataset}.")
    else: 
        try:
            hyperparameters_path = hyperparameters_path[0]
        except:
            logger.info(f"Hyperparameters for {best_hp_method} and {dataset} not found.")
            logger.info(f"hyperparameters_path: {hyperparameters_path}")
            raise FileNotFoundError(
                f"Hyperparameters for {best_hp_method} and {dataset} not found."
            )
    with open(hyperparameters_path) as f:
        hyperparameters = yaml.safe_load(f)
    return hyperparameters

In [ ]:
# setup an untrained LGBM with hyperparameters from tuning
def model_setup(dataset, method, device='gpu', verbose=-1):
    hyperparameters = get_hyperparameters(dataset, method)
    task = benchmark_datasets[dataset].task
    return LGBM(
        task=task,
        device=device,
        verbose=verbose,
        n_jobs=-1,
        **hyperparameters
    )

### Sort and Slice Importance

#### Sort and Slice Functions

In [ ]:
# get best sort and slice settings for a dataset
def get_sort_and_slice_settings(dataset):
    best_method = get_method_to_plot(dataset, "sns")
    best_method = best_method.split('_')
    radius = int(best_method[2])
    fpsize = int(best_method[-1])
    logger.info(f"Radius: {radius}; FP size: {fpsize}")
    return radius, fpsize

In [ ]:
# setup sort and slice generator
def setup_sort_and_slice(radius, save_img=False, **kwargs,):
    generator = MorganGenerator(radius=radius)
    generator = SortAndSlice(
        generator=generator,
        verbose=True,
        save_img=save_img,
        **kwargs,
    )
    return generator

In [ ]:
# load sort and slice generator for a dataset
def load_sort_and_slice(dataset):
    radius, fpsize = get_sort_and_slice_settings(dataset)
    return setup_sort_and_slice(radius=radius, fpsize=fpsize)

In [ ]:
# calculate substructures and pickle
def extract_substructures(sort_and_slice):
    substructures = {
        k: {
            "image": v["img"], 
            "atom": v["central_atom"], 
            "aromatic": v["aromatic"],
            "ring": v["ring"],
        } 
        for k, v in sort_and_slice.identifiers.items()
    }
    substructures["atom_colors"] = sort_and_slice.atom_colors
    substructures["ring_color"] = sort_and_slice.ring_color
    substructures["aromatic_color"] = sort_and_slice.aromatic_color
    return substructures

def save_substructures(dataset, max_radius=2, **kwargs):
    molecules, _, _ = setup_dataset(dataset)
    generator = setup_sort_and_slice(max_radius, save_img=True, **kwargs)
    generator.update(molecules=molecules)
    substructures = extract_substructures(generator)
    parent = TEMP_DIR / "sort_and_slice"
    parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        substructures,
        parent / f"{dataset}_{max_radius}_substructures.pt"
    )

In [ ]:
# load substructures from file
def load_substructures(dataset, max_radius=2, rerun=False, **kwargs):
    logger.info(f"Loading substructures for {dataset} with max radius {max_radius}.")
    logger.info(f"Rerun: {rerun}")
    logger.info(f"kwargs: {kwargs}")
    substructures_path = TEMP_DIR / "sort_and_slice" / f"{dataset}_{max_radius}_substructures.pt"
    if not substructures_path.exists() or rerun:
        save_substructures(dataset, max_radius=max_radius, **kwargs)
    return torch.load(
        substructures_path,
        map_location="cpu",
        weights_only=False
    )

#### Run Sort and Slice Importance

In [ ]:
# run importance calculations and output as a dataframe
def run_sort_and_slice_importance(
    dataset, repeats = 5, scorer=None,
    random_state=42, n_jobs=-1,
):
    molecules, y, splits = setup_dataset(dataset)
    substructures = load_substructures(dataset)
    sort_and_slice = load_sort_and_slice(dataset)

    out = {
        "Substructure": [],
        "Fold": [],
        "Importance": [],
        "Central Atom": [],
    }
    if scorer is None:
        if benchmark_datasets[dataset].task == "classification":
            scorer = "average_precision"
        else: scorer = "r2"

    logger.info(f"Running Sort and Slice importance for {dataset} with scorer {scorer}.")

    for i, (train, test) in enumerate(splits):
        train_mols, test_mols = molecules[train], molecules[test]
        train_y, test_y = y[train], y[test]
        sort_and_slice.clear()
        train_X = sort_and_slice(train_mols)
        test_X = sort_and_slice(test_mols)
        model = model_setup(dataset, "sns")
        model.fit(train_X, train_y)
        
        substructure_importance = permutation_importance(
            model,
            test_X, test_y,
            n_repeats=repeats,
            random_state=random_state,
            scoring=scorer,
            n_jobs=n_jobs,
        )
        out["Fold"].extend([i]*len(sort_and_slice.decoder)*repeats)
        for j in range(len(sort_and_slice.decoder)):
            substructure_id = sort_and_slice.decoder[j]
            out["Substructure"].extend([substructure_id]*repeats)
            out["Central Atom"].extend([substructures[substructure_id]['atom']]*repeats)
            out["Importance"].extend(substructure_importance["importances"][j, :])
    return pd.DataFrame(out)

In [ ]:
# loop over datasets and run/load S&S importance
if runner:
    importance_path = IMPORTANCE_RESULTS
    sort_and_slice_importances = {}
    for dataset in datasets_to_evaluate:
        Path(importance_path / dataset).mkdir(parents=True, exist_ok=True)
        save_path = importance_path / dataset / f"{dataset}_sort_and_slice_importance.csv"
        if save_path.exists():
            sort_and_slice_importances[dataset] = pd.read_csv(save_path)
        else:
            df = run_sort_and_slice_importance(dataset)
            df.to_csv(save_path, index=False)
            sort_and_slice_importances[dataset] = df

### PT-GIN Importance

#### PT-GIN Importance Functions

In [ ]:
# load best pre-trained gin on tuning splits
def load_pt_gin(dataset, device="cuda", **kwargs):
    model_name = get_method_to_plot(dataset, "pt_gin")
    model_path = [i for i in MODEL_DIR.rglob(f"{model_name}.pt")]
    if len(model_path) > 1:
        raise ValueError(
            f"More than one model found for {dataset}.\
                \n\tModel name: {model_name}\
                \n\tModel paths: {model_path}"\
        )
    logger.info(f"PT-GIN Model: {model_name}")
    model = PreTrainedFeaturizer(
        transform_kwargs={
            "path": model_path[0],
            "gnn": True,
            "asarray": True,
            "device": device,
            **kwargs
        },
    )
    return model

#### Run PT-GIN Importance

In [ ]:
# run importance calculations and output as a dataframe
def run_pt_gin_importance(
    dataset, repeats=5, scorer=None,
    random_state=42, n_jobs=-1, perm_type="dist",
):
    molecules, y, splits = setup_dataset(dataset)
    substructures = load_substructures(dataset)
    gnn = load_pt_gin(dataset, layer_pool_type="concat")
    decoder = {v: k for k, v in gnn.transform.featurizer.node_types.items()}
    cache_path = TEMP_DIR / f"{dataset}_importance_cache.csv"

    if cache_path.exists():
        logger.info(f"Cache found at {cache_path}. Loading existing results.")
        out = pd.read_csv(cache_path)
        out = out.to_dict(orient="list")
    else:
        logger.info("No cache found. Starting from beginning.")
        out = {
            "Substructure": [],
            "Fold": [],
            "Importance": [],
            "Central Atom": [],
        }

    if scorer is None:
        if benchmark_datasets[dataset].task == "classification":
            scorer = "average_precision"
        else: scorer = "r2"

    logger.info(f"Running PT-GIN importance for {dataset} with scorer {scorer}.")
    
    with tqdm(total=len(splits), desc=f"Substructure importance on {dataset}") as pbar:
        for i, (train, test) in enumerate(splits):
            if i in out["Fold"]:
                logger.info(f"Fold {i} already processed. Skipping.")
                pbar.update(1)
                logger.info(": " + str(pbar))
                continue
            train_mols, test_mols = molecules[train], molecules[test]
            train_y, test_y = y[train], y[test]
            train_X = gnn(train_mols)
            test_graphs = gnn.initial_embed(test_mols, keep_tokens=True)
            model = model_setup(dataset, "pt_gin",)
            model.fit(train_X, train_y)
            del train_X
            torch.cuda.empty_cache()
            substructure_importance = gnn_importance.token_importance(
                model, gnn, test_graphs, test_y,
                n_repeats=repeats, 
                scorer=scorer, 
                random_state=random_state,
                n_jobs=n_jobs,
                perm_type=perm_type,
                batch_size=128 if "MUV" in dataset else 512
            )
            
            out["Fold"].extend([i]*len(decoder)*repeats)

            for j in range(len(decoder)):
                substructure_id = decoder[j]

                out["Substructure"].extend([substructure_id]*repeats)
                if substructure_id in substructures:
                    central_atom = substructures[substructure_id]['atom']
                else: central_atom = None
                out["Central Atom"].extend([central_atom]*repeats)
                out["Importance"].extend(substructure_importance["importances"][j, :])
            pd.DataFrame(out).to_csv(cache_path, index=False)
            pbar.update(1)
            logger.info(": " + str(pbar))
    cache_path.unlink(missing_ok=True)
    out = pd.DataFrame(out)
    return out

In [ ]:
# loop over datasets and run/load S&S importance
if runner:
    importance_path = IMPORTANCE_RESULTS
    pt_gin_importances = {}
    for dataset in datasets_to_evaluate:
        logger.info(f"Running PT-GIN importance for {dataset}")
        Path(importance_path / dataset).mkdir(parents=True, exist_ok=True)
        save_path = importance_path / dataset / f"{dataset}_pt_gin_importance.csv"
        if save_path.exists():
            pt_gin_importances[dataset] = pd.read_csv(save_path)
        else:
            df = run_pt_gin_importance(dataset)
            df.to_csv(save_path, index=False)
            pt_gin_importances[dataset] = df

### Importance Plotting

#### Plotting functions

In [ ]:
# plot a swarm plot of importances
# input importances as df with cols for "Substructure", "Fold", and "Importance"
# set marker based on train-test fold
def importance_swarm(df, top_substructures, ax, **kwargs):
    markers = ["^", "X", "P", "o", "v"]
    df["markers"] = df["Fold"].apply(lambda x: markers[x])
    swarm_ax = sns.swarmplot(
        df[df["Substructure"].isin(top_substructures)], 
        x="Substructure", y="Importance", hue="Fold", 
        order=top_substructures, 
        size=6, dodge=False,
        ax=ax,
    )
    swarm_colors = np.unique(swarm_ax.collections[0].get_facecolor(), axis=0)
    marker_map = {}
    for i, c in enumerate(swarm_colors):
        marker_obj = MarkerStyle(markers[i])
        marker_map[str(c)] = marker_obj.get_path().transformed(marker_obj.get_transform())
    
    collections = swarm_ax.collections
    swarm_colors = np.unique(swarm_ax.collections[0].get_facecolor(), axis=0)
    for col in collections:
        paths = []
        for c in col.get_facecolor():
            c = str(c)
            if c in marker_map:
                paths.append(marker_map[c])
        col.set_paths(paths)
        col.set_facecolor("black")
    return swarm_ax

In [ ]:
# add hatches to aromatics
def add_aromatic_hatches(ax, top_substructures, substructures):
    patches = [i for i in ax.get_children() if isinstance(i, PolyCollection)]
    for patch, substruc in zip(patches, top_substructures):
        if substruc == "UNK": continue
        if substructures[int(substruc)]["aromatic"]:
            patch.set_hatch("//")
        elif substructures[int(substruc)]["ring"]:
            patch.set_hatch("\\\\")

In [ ]:
def add_unk_to_substructures(substructures):

    if "UNK" not in substructures:
        example = list(substructures.values())[0]["image"]
        example = np.asarray(example)
        font = plt.rcParams['font.sans-serif'][2]
        font_path = findfont(FontProperties(family=font))
        font = ImageFont.truetype(font_path, 35)
        
        w, h = example.shape[1], example.shape[0]
        img = Image.new('RGB', (w, h), color=(255,255,255))
        d = ImageDraw.Draw(img)
        text = "UNK"
        bbox = d.textbbox((0,0), text, font=font)
        text_w, text_h = bbox[2] - bbox[0], bbox[3] - bbox[1]

        # Compute centered position
        x_pos = (w - text_w) / 2
        y_pos = (h - text_h) / 2

        d.text((x_pos,y_pos), text, fill=(0, 0, 0), font=font, align="left")
        substructures["UNK"] = {
            "image": img
        }
    return substructures
        

In [ ]:
# add substructure images to y axis
def add_substructure_img(
    ax, substructures, 
    y_pad=0.12, zoom=0.6,
    img_pad=0.5,
):
    substructures = add_unk_to_substructures(substructures)
    ymin, _ = ax.get_ylim()
    y_pos = ymin - y_pad
    for i in ax.get_xticklabels():
        lab = i._text
        x = float(i.get_position()[0])
        y = float(y_pos)
        if lab != "UNK":
            lab = int(lab)
        try:
            img = substructures[lab]["image"]
        except Exception as e:
            print(i, lab)
            print(e)
            break
        img = OffsetImage(np.asarray(img), zoom=zoom,)
        ab = AnnotationBbox(
            img, (x, y),
            pad=img_pad,
            annotation_clip=False,
        )
        ax.add_artist(ab)

In [ ]:
# plot importances on a violin
# input importances as df with cols for "Substructure", "Fold", and "Importance"
def importance_violin(
    df, substructures, 
    n_violins=10, ax=None, 
    swarm_kwargs={}, 
    metric=None,
    **kwargs
):
    mean_importance = df.groupby("Substructure")["Importance"].mean()
    top_substructures = mean_importance.sort_values(ascending=False).index[:n_violins]

    ax = importance_swarm(df, top_substructures, ax=ax, **swarm_kwargs)
    
    ax = sns.violinplot(
        df[df["Substructure"].isin(top_substructures)], 
        x="Substructure", y="Importance", hue="Central Atom",
        order=top_substructures, 
        density_norm="width", alpha=0.8,
        ax=ax,
        **kwargs
    )
    if metric is not None:
        ax.set_ylabel(metric_display_names[metric])
    add_aromatic_hatches(ax, top_substructures, substructures)
    
    return ax

In [ ]:
# add fold and central atom legends
def remake_legend(
    fig, axes, folds,
    ring_color=(1,1,1), aromatic_color=(1,1,1),
    fold_kwargs=dict(
        loc="lower center",
        bbox_to_anchor=(0.5, 0.99),
        prop={"size": 22},
    ),
    atom_kwargs=dict(
        loc="upper center",
        bbox_to_anchor=(0.5, 0.99),
        prop={"size": 22}
    ),
):
    fold_handles = {}
    atom_handles = {}
    markers = ["^", "X", "P", "o", "v"]
    folds = [str(i) for i in range(folds)]
    for ax in axes:
        handles, labels = ax.get_legend_handles_labels()
        for h, l in zip(handles, labels):
            if l not in fold_handles:
                if l in folds:
                    h = plt.Line2D(
                        [0], [0],
                        marker=markers[int(l)],
                        ls="none", mec="none", color='black', 
                        markersize=10,
                        label=l,
                    )
                    fold_handles[l] = h
            if l not in atom_handles:
                if l not in folds:
                    atom_handles[l] = h

        ax.get_legend().remove()
    atom_handles["aromatic"] = Patch(
        edgecolor="black",
        facecolor="white",
        hatch="///",
        label="Aromatic"
    )
    atom_handles["ring"] = Patch(
        edgecolor="black",
        facecolor="white",
        hatch="\\\\\\",
        label="Ring"
    )
    peripheral_handles = []
    if not np.all(ring_color):
        peripheral_handles.append(
            Patch(
                edgecolor="black",
                facecolor=ring_color,
                label="Ring"
            )
        )
    if not np.all(aromatic_color):
        peripheral_handles.append(
            Patch(
                edgecolor="black",
                facecolor=aromatic_color,
                label="Aromatic"
            )
        )
    
    if "ncol" not in fold_kwargs:
        fold_kwargs["ncol"] = len(folds)
    if "ncol" not in atom_kwargs:
        atom_kwargs["ncol"] = len(atom_handles)
    fig.legend(
        handles=fold_handles.values(),
        title="Folds",
        **fold_kwargs,
    )
    fig.legend(
        handles=atom_handles.values(),
        title="Central Atom",
        **atom_kwargs
    )
    if len(peripheral_handles) > 0:
        fig.legend(
            handles=peripheral_handles,
            loc="upper center",
            ncol=len(peripheral_handles),
            bbox_to_anchor=(0.5, 0.99),
            title="Peripheral Atoms",
            prop={"size": 22}
        )

#### Run Importance Plotting

In [ ]:
# set sns theme
sns.set_theme(
    rc={
        "figure.figsize": (20, 20),
        "axes.labelsize": 18,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "axes.titlesize":20,
        "figure.subplot.hspace": 0.6,
        "figure.subplot.wspace": 0.3,
        "legend.title_fontsize": 30,
    }
)

In [ ]:
substructure_kwargs = {
    'FreeSolv': {
        "sns": {},
        "gin": {},
    }, 
    'Solu': {
        "sns": {"img_kwargs": {"y_pad": 0.02}},
        "gin": {"img_kwargs": {"y_pad": 0.02}},
    }, 
    'Human_CLint': {
        "sns": {"img_kwargs": {"y_pad": 0.025}},
        "gin": {"img_kwargs": {"y_pad": 0.025}},
    }, 
    'SR_ARE': {
        "sns": {"img_kwargs": {"y_pad": 0.015}},
        "gin": {"img_kwargs": {"y_pad": 0.015}},
    },
    'MUV858': {
        "sns": {"img_kwargs": {"y_pad": 0.0008}},
        "gin": {"img_kwargs": {"y_pad": 0.0008}},
    }
}

In [ ]:
# run importance violin plotting
if runner:
    break_loop = False
    
    for dataset in datasets_to_evaluate:
        if benchmark_datasets[dataset].task == "regression":
            metric="r2"
        else: metric = "aucpr"
        sort_and_slice_imp = sort_and_slice_importances[dataset]
        pt_gin_imp = pt_gin_importances[dataset]
        substructures = load_substructures(dataset)
        atom_colors = substructures["atom_colors"]
        atom_colors["UNK"] = (1.0, 1.0, 1.0)  # set unknown atom color
        ring_color = substructures["ring_color"]
        aromatic_color = substructures["aromatic_color"]

        fig, axes = plt.subplots(2,1)
        axes = axes.ravel()

        axes[0] = importance_violin(
            sort_and_slice_imp, substructures, ax=axes[0],
            palette=atom_colors, inner=None, metric=metric,
            **substructure_kwargs[dataset]["sns"]
        )
        sns_ylims = axes[0].get_ylim()
        pt_gin_imp.loc[pt_gin_imp["Substructure"] == "UNK", "Central Atom"] = "UNK"
        axes[1] = importance_violin(
            pt_gin_imp, substructures, ax=axes[1],
            palette=atom_colors, inner=None, metric=metric,
            **substructure_kwargs[dataset]["gin"]
        )
        gin_ylims = axes[1].get_ylim()
        ylim = (min(sns_ylims[0], gin_ylims[0]), max(sns_ylims[1], gin_ylims[1]))
        axes[0].set_ylim(ylim)
        axes[1].set_ylim(ylim)

        add_substructure_img(axes[0], substructures, **substructure_kwargs[dataset]["sns"].get("img_kwargs", {}))
        add_substructure_img(axes[1], substructures, **substructure_kwargs[dataset]["gin"].get("img_kwargs", {}))

        
        remake_legend(
            fig, axes, folds=sort_and_slice_imp.Fold.max() + 1,
            ring_color=ring_color, aromatic_color=aromatic_color,
        )
        for i, ax in enumerate(axes):
            ax.set_title(f"({ascii_lowercase[i]})", loc="left", pad=20)
            ax.set_xlabel("Substructure", labelpad=100)
            render_box(fig, [ax])
        
        if save_figs:
            if dataset == "FreeSolv":
                sub_dir = "main_body"
            else: sub_dir = "supplementary"
            plt.savefig(
                FIGURES_DIR / sub_dir / f"{dataset}_substructure_importance",
                bbox_inches="tight",
            )
        else:
            plt.show()
            break_loop = True
        plt.close()
        if break_loop: break

### FreeSolv Distribution

In [ ]:
if runner:
    molecules, y, _ = setup_dataset("FreeSolv")

    oh_smarts = "[OX2H]"
    pattern = Chem.MolFromSmarts(oh_smarts)

    df = pd.DataFrame({"$\\Delta \\mathrm{G_{solv}}$": y, "molecule": molecules})
    df["OH Present"] = df["molecule"].apply(
        lambda x: False if x is None else False if len(x.GetSubstructMatches(pattern)) == 0 else True
    )


In [ ]:
# set sns theme
if runner:
    sns.set_theme(
        rc={
            "figure.figsize": (10, 20),
            "axes.labelsize": 20,
            "xtick.labelsize": 16,
            "ytick.labelsize": 16,
            "axes.titlesize":20,
            "figure.subplot.hspace": 0.6,
            "figure.subplot.wspace": 0.3,
            "legend.title_fontsize": 20,
            "legend.fontsize":16
        }
    )
    fig, axes = plt.subplots(1,1)
    ax = sns.violinplot(
        df, y="$\\Delta \\mathrm{G_{solv}}$", hue="OH Present", 
        split=True, gap=0.05,
    )
    render_box(fig, [ax])
    if save_figs:
        plt.savefig(
            FIGURES_DIR / "extended" / "freesolv_oh_bias",
            bbox_inches="tight",
        )
    else: 
        plt.show()
    plt.close()